# Pull deps

In [1]:
import sys
!{sys.executable} -m pip install datasets huggingface_hub scipy librosa "numpy<2.3"


Defaulting to user installation because normal site-packages is not writeable
ERROR: Error while checking for conflicts. Please file an issue on pip's issue tracker: https://github.com/pypa/pip/issues/new
Traceback (most recent call last):
  File "/home/nicolas/.local/lib/python3.13/site-packages/pip/_internal/commands/install.py", line 587, in _determine_conflicts
    return check_install_conflicts(to_install)
  File "/home/nicolas/.local/lib/python3.13/site-packages/pip/_internal/operations/check.py", line 116, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/nicolas/.local/lib/python3.13/site-packages/pip/_internal/operations/check.py", line 58, in create_package_set_from_installed
    package_set[name] = PackageDetails(dist.version, dependencies)
                                       ^^^^^^^^^^^^
  File "/home/nicolas/.local/lib/python3.13/site-packages/pip/_internal/metadata/

# Setup HF's storage manually (I need a separated HDD)

In [1]:
from pathlib import Path

import os


In [2]:
CACHE_DIR = Path("./.cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Set root cache for all HF libs (datasets, hub, transformers, etc.)
_p = str(CACHE_DIR.resolve())
os.environ["HF_HOME"] = _p

print("Set HF home dir to", _p)


Set HF home dir to /mnt/external/datasets/.cache


# Setting up

In [4]:
from subprocess import Popen, PIPE, CalledProcessError, run
from huggingface_hub import HfApi, list_repo_files
from datasets import Dataset, Audio, ClassLabel, load_dataset, concatenate_datasets, Value

from datasets.dataset_dict import DatasetDict
from pathlib import Path
from IPython.display import Audio as IPA, display
from concurrent.futures import ThreadPoolExecutor, as_completed
import datasets

import gc
import os
import sys
import glob
import librosa
import datasets
import numpy as np
import scipy.io as sio
import pandas as pd


/home/nicolas/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
REPO_ID = "Hibou-Foundation/datian"
LOCAL_DIR = Path("./prepared_dataset")
REPO_TYPE = "dataset"


In [6]:
#The following DSs rely on these values
#geronimobasso/drone-audio-detection-samples
#yehiellevi/dataset-balanced-n-weighted-final
#Mixed datasets also follow this convention (at least the ones we currently have)
CLASSES_INV = {0:"other", 1: "drone"}
CLASSES = {"other": 0, "drone": 1}


In [7]:
dl_dir = Path("./downloaded_datasets")
local_tmp_dir = Path("./staging")


In [8]:
dl_dir.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
local_tmp_dir.mkdir(parents=True, exist_ok=True)


In [9]:
def easy_display_audio(audio, sample_rate):
    if np.max(np.abs(audio)) > 1:
        audio = audio / np.max(np.abs(audio))

    # Display the audio player
    display(IPA(data=audio, rate=int(sample_rate.item())))


# Sources

In [10]:
download_args = [
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%202%20Dataset.zip", "--output", f"{dl_dir}/scenario2.zip"],
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%201%20Dataset.zip", "--output", f"{dl_dir}/scenario1.zip"],
    ["https://github.com/DroneDetectionThesis/Drone-detection-dataset/archive/refs/heads/master.zip", "--output", f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset.zip"],
    ["https://uc90e6a3b40e150d234227daf272.dl.dropboxusercontent.com/cd/0/get/C7eK5DuGCpRv4uV9pfWTsFUjgjNtBXCZpqBEFPVxcwY-jwBmfseQgQotPOZ3iBQQfdKJXZSjhkXlQVL5PclGCNhR7Sd3uxEBg9IeuD5RoBWJY7wH92mlASk-I-Y8Xsyynx18jNUG7gU7QP7XI5THb_zm/file?_download_id=1637074187773826591426354030248830796317808430536170451142051367&_log_download_success=1&_notify_domain=www.dropbox.com&dl=1", "--output", f"{dl_dir}/aira-uas.tar.gz"],
    ["https://zenodo.org/records/15391924/files/Microphone_array.zip?download=1", "--output", f"{dl_dir}/UaVirBASE.zip"],
    ["https://data.nasa.gov/docs/datasets/rfk401li/small_uav_acoustics.zip", "--output", f"{dl_dir}/nasa.small_uav_flyover_acoustics.zip"],
    ["https://www.kaggle.com/api/v1/datasets/download/yehiellevi/dataset-balanced-n-weighted-final", "--output", f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final.zip"],
]

file_names = [
    "scenario2.zip",
    "scenario1.zip",
    "UaVirBASE.zip",
    "DroneDetectionThesis.Drone-detection-dataset.zip",
    "aira-uas.tar.gz",
    "nasa.small_uav_flyover_acoustics.zip",
    "yehiellevi.dataset-balanced-n-weighted-final.zip",
]


In [11]:
hf_mixed_sources = [
    "geronimobasso/drone-audio-detection-samples",
    "Hibou-Foundation/df_462700_2",
]

# ("ahlab-drone-project/DroneAudioSet", ['drone-only', 'drone-with-source', 'ground-truth', 'source-only'])
# drone-with-source has been selected because of its noisy mixtures.

hf_drone_sources = [
    ("ahlab-drone-project/DroneAudioSet", ["drone-with-source"]),
]

#    "sps44/fsdnoisy18k",
hf_other_sources = [
    "agkphysics/AudioSet",
    "FluidInference/musan",
]


# Download DS archives

In [13]:
last_dl = 0


In [14]:
def run_command(cmd, dry=False):
    if dry:
        print(" ".join(cmd))
        return None

    with Popen(cmd, stdout=PIPE, bufsize=1, universal_newlines=True) as p:
        for line in p.stdout:
            print(line, end='') # process line here

        if p.returncode != 0 and p.returncode is not None:
            raise CalledProcessError(p.returncode, p.args)

        if p.returncode is not None:
            print("Exited with code", p.returncode)


In [15]:
def unpack_archives(files, dry=True):
    for f in files:
        print("Unpacking", f)
        if f.endswith(".zip"):
            run_command(["unzip", str(dl_dir / f), "-d", f"{dl_dir}/{f[:-4]}"], dry)
        elif f.endswith(".tar.gz"):
            run_command(["mkdir", "-p", f"{dl_dir}/{f[:-7]}"], dry)
            run_command(["tar", "-zxvf", str(dl_dir / f), "-C", f"{dl_dir}/{f[:-7]}", "--strip-components=1"], dry)


In [26]:
for i in range(last_dl, len(download_args)):
    run_command(["curl"] + download_args[i])
    last_dl = i


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0 3918M    0 2414k    0     0   762k      0  1:27:41  0:00:03  1:27:38  762k

KeyboardInterrupt: 

In [17]:
unpack_archives(file_names, dry=False)


Unpacking yehiellevi.dataset-balanced-n-weighted-final.zip
Archive:  downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final.zip
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/audio_metadata_shuffled.csv  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/0_20241210_105052382___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/0_20241210_105942761___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/1-100038-A-140___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/1-100038-A-143___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/1-100210-A-363___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/1-101296-A-192___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/fold1/1-10

# Pull in the HF datasets

In [47]:
hf_other = [load_dataset(ds) for ds in hf_other_sources]


Repo card metadata block was not found. Setting CardData to empty.


In [16]:
hf_mixed = [load_dataset(ds) for ds in hf_mixed_sources]


In [12]:
hf_drone = [load_dataset(ds) if ds is str else [load_dataset(ds[0], conf) for conf in ds[1]] for ds in hf_drone_sources]


Using the latest cached version of the dataset since ahlab-drone-project/DroneAudioSet couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'drone-with-source' at /mnt/external/datasets/.cache/datasets/ahlab-drone-project___drone_audio_set/drone-with-source/0.0.0/e794007062e2d2ad29262a5795b60d09f2b345c4 (last modified on Wed Feb 25 14:36:46 2026).


# Process the downloaded archives

In [9]:
def gen_aira_uas():
    data_root = f"{dl_dir}/aira-uas/"

    protos = [f"Protocol{i}" for i in range(1, 4)]
    infos = {"Protocol1": ([i for i in range(4, 16)], 18), "Protocol2": ([18], 3), "Protocol3": ([21], 3)}
    
    drones = []
    others = []
    elements = []
    
    j = 0
    for i in range(1, 4):
        subp = f"Protocol{i}"
        for k in range(infos[subp][1]):
            path = data_root + subp + f"/Recording{j}"
            if j in infos[subp][0]:
                drones += glob.glob(path + "/*.wav")
            else:
                others += glob.glob(path + "/*.wav")
            j += 1

    for f, label in [(f, CLASSES["drone"]) for f in drones] + [(f, CLASSES["other"]) for f in others]:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": label, "src": "aira"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": label, "src": "aira"})

    return Dataset.from_list(elements)


In [10]:
def gen_ddd():
    data_root = f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset/Drone-detection-dataset-master/Data/Audio/"

    drones = glob.glob(data_root + "DRONE_*.wav")
    others = glob.glob(data_root + "BACKGROUND_*.wav") + glob.glob(data_root + "HELICOPTER_*.wav") + glob.glob(data_root + "DRONE_*.wav")

    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"]})

    for f in others:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})

    return Dataset.from_list(elements)


In [11]:
def gen_dalrd():
    data_root_1 = f"{dl_dir}/scenario1/Scenario 1 Dataset/"
    data_root_2 = f"{dl_dir}/scenario2/Scenario 2 Dataset/"
    
    drones = glob.glob(data_root_1 + "*.wav", recursive=True) + glob.glob(data_root_2 + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
    
    return Dataset.from_list(elements)


In [12]:
def gen_uavirbase():
    data_root = f"{dl_dir}UaVirBASE/Microphone_array/"
    drones = glob.glob(data_root + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio())
    

In [13]:
def gen_yehiellevi():
    data_root = f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final/"
    df = pd.read_csv(data_root + "audio_metadata_shuffled.csv", sep=",")
    
    elements = []
    
    for _, r in df.iterrows():
        path = f"{data_root}/fold{r['fold']}/{r['slice_file_name']}"
        start, end = r["start"], r["end"]
        data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": r["classID"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": r["classID"], "src": "yehiellevi"})
    
    return Dataset.from_list(elements)
    

# NASA's DS is a bit more complicated than the others.

In [14]:
def load_mat(filepath: str) -> dict:
    """Load a .mat file, trying scipy first (v5), then h5py for v7.3."""
    try:
        mat = sio.loadmat(filepath, squeeze_me=True, struct_as_record=False)
        return mat
    except NotImplementedError:
        # v7.3 HDF5-based .mat files
        try:
            import h5py
        except ImportError:
            print("ERROR: h5py is required for MATLAB v7.3 files. Install with: pip install h5py")
            sys.exit(1)

        def hdf5_to_dict(h5obj):
            result = {}
            for key, val in h5obj.items():
                if isinstance(val, h5py.Dataset):
                    result[key] = val[()]
                elif isinstance(val, h5py.Group):
                    result[key] = hdf5_to_dict(val)
            return result

        import h5py
        with h5py.File(filepath, "r") as f:
            return hdf5_to_dict(f)


In [15]:
def get_field(struct, field: str):
    """Retrieve a field from either a scipy mat_struct or a plain dict."""
    if isinstance(struct, dict):
        return struct[field]
    return getattr(struct, field)


In [16]:
def extract_acoustic_data(mat: dict, mic_channel: int):
    """
    Extract acoustic pressure and UTC time from the 'acoustics' structure.

    Returns
    -------
    utc_time : np.ndarray  (n,)   seconds past midnight, UTC
    pressures : np.ndarray (n, num_mics)  incident pressures in Pascals
    selected_channel_pressure : np.ndarray (n,)  pressure for chosen mic
    sample_rate : float  Hz
    """
    acoustics = mat["acoustics"]

    utc_time   = np.asarray(get_field(acoustics, "utc_time"), dtype=float).ravel()
    pressures  = np.asarray(get_field(acoustics, "incident_pascals"), dtype=float)

    # Ensure 2-D: (n_samples, n_mics)
    if pressures.ndim == 1:
        pressures = pressures[:, np.newaxis]

    n_mics = pressures.shape[1]
    if mic_channel >= n_mics:
        raise ValueError(
            f"Requested channel {mic_channel} but data only has {n_mics} mic(s) "
            f"(0-based indexing)."
        )

    selected_pressure = pressures[:, mic_channel]

    # Sampling rate from mean time step
    dt          = np.mean(np.diff(utc_time))
    sample_rate = 1.0 / dt

    return utc_time, pressures, selected_pressure, sample_rate


In [17]:
def extract_vehicle_data(mat: dict):
    """Extract RTK and GPS vehicle position data."""
    vd = mat.get("vehicle_data")
    if vd is None:
        return None

    result = {}
    for field in ("rtk_utc_time", "rtk_ned_meters", "rtk_status",
                  "gps_utc_time", "gps_ned_meters"):
        try:
            result[field] = np.asarray(get_field(vd, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [18]:
def extract_met_data(mat: dict):
    """Extract meteorological data if available."""
    met = mat.get("met_data")
    if met is None:
        return None

    result = {}
    for field in ("temperature_celsius", "windspeed_knots", "wind_direction", "utc_time"):
        try:
            result[field] = np.asarray(get_field(met, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [19]:
def print_summary(utc_time, pressures, selected_pressure, sample_rate,
                  mic_channel, vehicle_data, met_data):
    """Print a human-readable summary to the console."""
    print("=" * 60)
    print("  NASA Small UAV Acoustic Data Extraction")
    print("=" * 60)

    duration = utc_time[-1] - utc_time[0]
    print(f"\n[ACOUSTICS]")
    print(f"  Sample rate      : {sample_rate:.2f} Hz")
    print(f"  Num samples      : {len(utc_time)}")
    print(f"  Duration         : {duration:.2f} s  ({duration/60:.2f} min)")
    print(f"  Num mic channels : {pressures.shape[1]}")
    print(f"  Selected channel : {mic_channel} (0-based)")
    print(f"  Pressure range   : [{selected_pressure.min():.6f}, {selected_pressure.max():.6f}] Pa")
    print(f"  Peak SPL (A-weighted ref 20µPa): "
          f"{20 * np.log10(np.sqrt(np.mean(selected_pressure**2)) / 20e-6):.1f} dB")

    if vehicle_data:
        print(f"\n[VEHICLE DATA]")
        rtk_t = vehicle_data.get("rtk_utc_time")
        gps_t = vehicle_data.get("gps_utc_time")
        ned_rtk = vehicle_data.get("rtk_ned_meters")
        ned_gps = vehicle_data.get("gps_ned_meters")
        if rtk_t is not None:
            print(f"  RTK samples      : {len(rtk_t.ravel())}")
        if gps_t is not None:
            print(f"  GPS samples      : {len(gps_t.ravel())}")
        if ned_rtk is not None and ned_rtk.ndim == 2:
            r = np.sqrt(np.sum(ned_rtk**2, axis=1))
            print(f"  RTK max distance : {r.max():.2f} m from origin")
        if ned_gps is not None and ned_gps.ndim == 2:
            r = np.sqrt(np.sum(ned_gps**2, axis=1))
            print(f"  GPS max distance : {r.max():.2f} m from origin")

    if met_data:
        print(f"\n[METEOROLOGICAL DATA]")
        temp = met_data.get("temperature_celsius")
        wind = met_data.get("windspeed_knots")
        if temp is not None:
            print(f"  Avg temperature  : {np.nanmean(temp):.1f} °C")
        if wind is not None:
            print(f"  Avg wind speed   : {np.nanmean(wind):.1f} kn")
    else:
        print("\n[METEOROLOGICAL DATA] Not available")

    print()


In [20]:
def extract_nasa_data(mat_file, vehicle: str=None, channel: int=None):
    # Resolve mic channel
    if channel is not None:
        mic_channel = channel
    elif vehicle is not None:
        mic_channel = VEHICLE_MIC_CHANNEL[vehicle]
    else:
        mic_channel = 0
        print(f"No --vehicle or --channel specified; defaulting to channel 0.")

    # Load
    print(f"Loading: {mat_file}")
    mat = load_mat(mat_file)

    # Extract
    utc_time, pressures, selected_pressure, sample_rate = extract_acoustic_data(
        mat, mic_channel
    )

    return pressures[:, mic_channel], int(sample_rate)


In [21]:
def gen_nasa():
    data_root = f"{dl_dir}/nasa.small_uav_flyover_acoustics/data/"
    files = glob.glob(data_root + "*.mat")

    drones = []

    for f in files:
        basename = os.path.basename(f)
        if basename.startswith("cub_") or basename.startswith("hex_"):
            channel = 0
        else:
            channel = 2
        data, sr = extract_nasa_data(f, channel=channel)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                drones.append({"array": cast[i], "sampling_rate": sr})
        else:
            drones.append({"array": cast, "sampling_rate": sr})

    elements = [{"audio": f, "label": CLASSES["drone"], "src": "nasa"} for f in drones]
    return Dataset.from_list(elements)


# Load local DSs

In [22]:
def gen_gtrfd():
    data_root = f"./local_datasets/Archive/GTRFD/"
    
    drones = glob.glob(data_root + "*.wav", recursive=True)
    
    elements = []

    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "gtrfd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "gtrfd"})

    return Dataset.from_list(elements)


# Save what we've got so far.

In [23]:
_listed = [gen_aira_uas(), gen_ddd(), gen_dalrd(), gen_uavirbase(), gen_yehiellevi(), gen_nasa()] + [gen_gtrfd()]


/tmp/ipykernel_11137/405754235.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
/home/nicolas/.local/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_11137/405754235.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
/home/nicolas/.local/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/y6_hover_202.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_206.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_040.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_207.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_059.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/phantom_flyover_125.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_051.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_038.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_hover_198.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/y6_hover_205.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_201.mat
Loading: downloaded_datasets/nasa.small_uav

In [24]:
unified_locally = concatenate_datasets(_listed)


In [79]:
def gen_locally():
    return concatenate_datasets([gen_aira_uas(), gen_ddd(), gen_dalrd(), gen_uavirbase(), gen_yehiellevi(), gen_nasa()] + [gen_gtrfd()])


In [80]:
unified_locally = gen_locally()


/tmp/ipykernel_4181/2556533235.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
/home/nicolas/.local/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/y6_hover_202.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_206.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_040.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_207.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_059.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/phantom_flyover_125.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_051.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/edge_flyover_038.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_hover_198.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/y6_hover_205.mat
Loading: downloaded_datasets/nasa.small_uav_flyover_acoustics/data/hex_flyover_201.mat
Loading: downloaded_datasets/nasa.small_uav

ValueError: The features can't be aligned because the key array of features {'array': List(Value('float64')), 'sampling_rate': Value('int64')} has unexpected type - List(Value('float64')) (expected either List(Value('float32')) or Value("null").

In [ ]:
gc.collect()

In [25]:
unified_locally.save_to_disk(str(local_tmp_dir) + "/local_gathering")


Saving the dataset (5/5 shards): 100%|██████| 21886/21886 [00:01<00:00, 12997.34 examples/s]


In [56]:
gc.collect()


501

In [12]:
wtf = datasets.load_from_disk(str(local_tmp_dir) + "/local_gathering")


In [54]:
print(unified_locally.features)


{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}


In [ ]:
def split_channels(batch):
    new_audios = []
    new_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        array = audio["array"]
        sr = audio["sampling_rate"]

        # Skip anything that is None or empty
        if array is None or array.size == 0:
            continue

        # Handle mono and multi-channel
        if array.ndim == 1 or array.shape[0] == 1:
            new_audios.append(audio)
            new_labels.append(label)
        else:
            for ch in range(array.shape[0]):
                new_audios.append({
                    "array": array[ch, :],
                    "sampling_rate": sr,
                })
                new_labels.append(label)

    return {"audio": new_audios, "label": new_labels}


# Uniformise the drone DSs

In [10]:
def ds_split_channels(ds):
    elements = []

    for i in range(len(ds)):
        elem = ds[i]
        sr = elem["audio"]["sampling_rate"]
        audios = np.array(elem["audio"]["array"]).T

        elements += [{"audio": {"sampling_rate": sr, "array": chan}} for chan in audios]

    ds = Dataset.from_list(elements).cast_column("audio", Audio())
    return ds.add_column("label", [CLASSES["drone"]] * len(ds))


In [11]:
def processs_trains(src, start=0):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items() if k.startswith("train_")]
    count = len(_elems)

    for i in range(start, count):
        k, ds = _elems[i]
        print(f"{i}/{count}")
        v = ds_split_channels(ds)
        v.save_to_disk(str(local_tmp_dir) + "/" + k)


In [38]:
def process_single_train(args):
    i, k, ds, count, local_tmp_dir = args
    print(f"{i}/{count}")
    v = ds_split_channels(ds)
    v.save_to_disk(str(local_tmp_dir) + "/" + k)
    return i, k

def processs_trains(src, start=0, num_threads=4):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items() if k.startswith("train_")]
    count = len(_elems)

    tasks = [
        (i, k, ds, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_single_train, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                i, k = future.result()
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")


In [36]:
processs_trains(hf_drone[0][0])


180/252
181/252
182/252
183/252


Saving the dataset (0/1 shards):   0%|                                                                                                                                                                            | 0/34 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|                                                                                                                                                                            | 0/34 [00:00<?, ? examples/s]


Saving the dataset (0/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 56.90 examples/s]

Saving the dataset (0/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 57.44 examples/s]


Saving the dataset (1/1 shards

184/252
185/252




Saving the dataset (1/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [01:53<00:00,  3.35s/ examples]


186/252
187/252


...

Saving the dataset (1/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [01:05<00:00,  1.94s/ examples]


250/252


Saving the dataset (1/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 62.99 examples/s]


251/252


Saving the dataset (1/1 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 63.54 examples/s]

Saving the dataset (1/1 shards): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 217.10 examples/s]


In [52]:
gc.collect()


535

In [41]:
def load_splitted_drones_ds(args):
    i, k, count, local_tmp_dir = args
    print(f"{i}/{count}")
    return datasets.load_from_disk(str(local_tmp_dir) + "/" + k)

def load_splitted_drones(src, start=0, num_threads=6):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items() if k.startswith("train_")]
    count = len(_elems)

    tasks = [
        (i, k, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    results = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(load_splitted_drones_ds, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")

    return concatenate_datasets(results)


In [42]:
unified_hf_drones = load_splitted_drones(hf_drone[0][0])


0/252
1/252
2/252
3/252
4/252
5/252
6/252
7/252
8/252
9/252
10/252
11/252
12/252
13/252
14/252
15/252
16/252
17/252
18/252
19/252
20/252
21/252
22/252
23/252
24/252
25/252
26/252
27/252
28/252
29/252
30/252
31/252
32/252
33/252
34/252
35/252
36/252
37/252
38/252
39/252
40/252
41/252
42/252
43/252
44/252
45/252
46/252
47/252
48/252
49/252
50/252
51/252
52/252
53/252
54/252
55/252
56/252
57/252
58/252
59/252
60/252
61/252
62/252
63/252
64/252
65/252
66/252
67/252
68/252
69/252
70/252
71/252
72/252
73/252
74/252
75/252
76/252
77/252
78/252
79/252
80/252
81/252
82/252
83/252
84/252
85/252
86/252
87/252
88/252
89/252
90/252
91/252
92/252
93/252
94/252
95/252
96/252
97/252
98/252
99/252
100/252
101/252
102/252
103/252
104/252
105/252
106/252
107/252
108/252
109/252
110/252
111/252
112/252
113/252
114/252
115/252
116/252
117/252
118/252
119/252
120/252
121/252
122/252
123/252
124/252
125/252
126/252
127/252
128/252
129/252
130/252
131/252
132/252
133/252
134/252
135/252
136/252
137/252
138/25

In [43]:
unified_hf_drones.save_to_disk(str(local_tmp_dir) + "/foreign_drone_gathering")


Saving the dataset (63/63 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8568/8568 [12:39<00:00, 11.28 examples/s]


In [ ]:
gc.collect()


# Uniformise the other DSs

In [ ]:
def gen_others():
    hf_other_simplified = []

    for opt in hf_other:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_other_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                cols = list(ds.features.keys())
                cols.remove("audio")
    
                hf_other_simplified.append(ds.remove_columns(cols).add_column("label", [CLASSES["other"]] * len(ds)))

    return concatenate_datasets(hf_other_simplified)


In [45]:
unified_hf_others = gen_others()


In [ ]:
gc.collect()


In [50]:
unified_hf_others.save_to_disk(str(local_tmp_dir) + "/foreign_other_gathering")


Saving the dataset (100/100 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36598/36598 [21:56<00:00, 27.80 examples/s]


In [ ]:
gc.collect()


# Uniformise the mixed DSs

In [17]:
def simplify_audio(elem):
    return {"label": elem["label"], "audio": {"sampling_rate": elem["sampling_rate"], "array": np.array(elem["audio"])}}

def simplify_audio(batch):
    return {
        "label": batch["label"],
        "audio": [
            {"sampling_rate": sr, "array": np.array(audio)}
            for sr, audio in zip(batch["sampling_rate"], batch["audio"])
        ]
    }


In [18]:
def gen_mixed():
    new_label = ClassLabel(names=["other", "drone"])
    hf_mixed_simplified = []

    for opt in hf_mixed:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_mixed_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                hf_mixed_simplified.append(ds)

    hf_mixed_pretty = []
    for ds in hf_mixed_simplified:
        if "sampling_rate" in ds.features:
            # Complicated matters
            hf_mixed_pretty.append(ds.map(simplify_audio, batched=True).remove_columns(["sampling_rate"]).cast_column("audio", Audio()).cast_column("label", new_label))
        else:
            hf_mixed_pretty.append(ds.cast_column("label", new_label))

    return concatenate_datasets(hf_mixed_pretty)
    

In [19]:
unified_hf_mixed = gen_mixed()


In [20]:
gc.collect()

1180

In [22]:
unified_hf_mixed.save_to_disk(str(local_tmp_dir) + "/foreign_mixed_gathering")


Saving the dataset (29/29 shards): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 643021/643021 [07:02<00:00, 1523.06 examples/s]


In [23]:
gc.collect()


155

# Now, we can properly set everything up for the final stage :)

In [10]:
stages = ["foreign_mixed_gathering", "foreign_drone_gathering", "foreign_other_gathering", "local_gathering"]


In [11]:
stages_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg) for stg in stages]



KeyboardInterrupt



In [ ]:
from datasets import Value


In [19]:
stages_ds[0] = stages_ds[0].cast_column("label", Value("int64"))


In [20]:
print([ds.features["label"] for ds in stages_ds])


[Value('int64'), Value('int64'), Value('int64'), Value('int64')]


In [21]:
output_ds = concatenate_datasets(stages_ds)


In [ ]:
output_ds.save_to_disk(LOCAL_DIR / "output", max_shard_size="300MB", num_proc=4)


Saving the dataset (295/325 shards):  91%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏            | 647896/708266 [20:46<40:04, 25.10 examples/s]

In [18]:
gc.collect()


435

In [25]:
past_output_ds = datasets.load_from_disk(LOCAL_DIR / "output")


FileNotFoundError: Directory prepared_dataset/output is neither a `Dataset` directory nor a `DatasetDict` directory.

In [27]:
from pathlib import Path
data_path = LOCAL_DIR / "output"
arrow_files = sorted(data_path.glob("data-*.arrow"))
num_complete = len(arrow_files)
print(f"Found {num_complete} shards: {arrow_files[-3:]}")  # Last 3


Found 306 shards: [PosixPath('prepared_dataset/output/data-00303-of-00325.arrow'), PosixPath('prepared_dataset/output/data-00304-of-00325.arrow'), PosixPath('prepared_dataset/output/data-00305-of-00325.arrow')]


In [28]:
from datasets import Dataset, concatenate_datasets, load_dataset
from pathlib import Path

data_path = LOCAL_DIR / "output"
total_shards = 325
arrow_files = sorted(data_path.glob("data-*.arrow"))
num_complete = len(arrow_files)


In [29]:
# Load completed shards (they have identical schemas)
complete_ds = Dataset.from_file(str(arrow_files[0]))  # Load first to get schema
for f in arrow_files[1:]:
    complete_ds = concatenate_datasets([complete_ds, Dataset.from_file(str(f))])

print(f"Loaded {len(complete_ds)} examples from {num_complete} shards")


Exception ignored in: <_io.BytesIO object at 0x7eff18a46340>
Traceback (most recent call last):
  File "/home/nicolas/.local/lib/python3.13/site-packages/datasets/table.py", line 108, in __init__
    recordbatch for recordbatch in table.to_batches() if len(recordbatch) > 0
BufferError: Existing exports of data: object cannot be re-sized


OSError: Expected to be able to read 139451400 bytes for message body, got 139448587

In [ ]:
# Generate missing shards 306-324 from original dataset
missing_shards = []
for i in range(num_complete, total_shards):
    shard = output_ds.shard(num_shards=total_shards, index=i, contiguous=True)
    shard_path = data_path / f"data-{i:05d}-of-{total_shards:05d}.arrow"
    shard.save_to_disk(shard_path.parent)  # Saves single shard as arrow dir
    missing_shards.append(Dataset.from_file(shard_path / "data-00000-of-00001.arrow"))

# Combine all
final_ds = concatenate_datasets([complete_ds] + missing_shards)
final_ds.save_to_disk(LOCAL_DIR / "output_complete")


# Just ensure every audio is mono, just in case.

In [18]:
def split_channels(example):
    audio = example["audio"]["array"]         # shape: [C, N] or [N]
    sr = example["audio"]["sampling_rate"]

    # Already mono -> keep it as-is
    if audio.ndim == 1 or audio.shape[0] == 1:
        return {"audio": [example["audio"]]}

    # Multi-channel -> create one entry per channel
    split_audios = []
    for ch in range(audio.shape[0]):
        split_audios.append({
            "array": audio[ch, :],
            "sampling_rate": sr
        })
    return {"audio": split_audios}


In [19]:
def split_channels(batch):
    new_audios = []
    new_labels = []  # keep any other metadata aligned

    for audio, label in zip(batch["audio"], batch["label"]):
        array = audio["array"]
        sr = audio["sampling_rate"]

        # Handle mono and multi-channel
        if array.ndim == 1 or array.shape[0] == 1:
            new_audios.append(audio)
            new_labels.append(label)
        else:
            for ch in range(array.shape[0]):
                new_audios.append({
                    "array": array[ch, :],
                    "sampling_rate": sr
                })
                new_labels.append(label)

    return {"audio": new_audios, "label": new_labels}


In [20]:
def split_channels(batch):
    new_audios = []
    new_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        array = np.array(audio["array"])
        sr = audio["sampling_rate"]

        # Skip anything that is None or empty
        if array is None or array.size == 0:
            continue

        # Handle mono and multi-channel
        if array.ndim == 1 or array.shape[0] == 1:
            new_audios.append(audio)
            new_labels.append(label)
        else:
            for ch in range(array.shape[0]):
                new_audios.append({
                    "array": array[ch, :],
                    "sampling_rate": sr,
                })
                new_labels.append(label)

    return {"audio": new_audios, "label": new_labels}


In [21]:
i = 0

In [26]:
ds = datasets.load_from_disk(str(local_tmp_dir) + "/" + stages[i])


In [23]:
ds = ds.map(split_channels, batched=True, num_proc=4, batch_size=8)


Map (num_proc=4): 100%|█████████████████████| 643021/643021 [13:49<00:00, 774.95 examples/s]


In [24]:
ds.save_to_disk(str(local_tmp_dir) + "/" + stages[i] + ".mono")


Saving the dataset (29/29 shards): 100%|███| 643021/643021 [04:49<00:00, 2219.15 examples/s]


In [27]:
print(len(ds))


643021


In [25]:
print(stages[i])


foreign_mixed_gathering


In [ ]:
for i in range(2, 3):
    print(stages[i])
    ds = datasets.load_from_disk(str(local_tmp_dir) + "/" + stages[i])
    ds.map(split_channels, batched=True, num_proc=4, batch_size=8)


foreign_other_gathering


Map (num_proc=4): 100%|████████████████████████| 36598/36598 [24:20<00:00, 15.14 examples/s]

In [23]:
ds_with_channels = output_ds.map(check_channels, batched=False, num_proc=6)


Map (num_proc=6):  91%|███████████████████  | 644221/708266 [18:28<01:50, 581.19 examples/s]


ArrowInvalid: offset overflow while concatenating arrays

In [ ]:
multi_channel_count = (ds_with_channels['n_channels'] > 1).sum()
total_rows = len(ds_with_channels)
print(f"Multi-channel rows: {multi_channel_count}/{total_rows} ({multi_channel_count/total_rows*100:.1f}%)")


In [11]:
stages = ["foreign_mixed_gathering", "foreign_drone_gathering", "foreign_other_gathering", "local_gathering"]


In [12]:
result_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg + ".mono") for stg in stages]


In [14]:
for ds in result_ds:
    print(ds.features)


{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': ClassLabel(names=['other', 'drone'])}
{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}
{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}
{'audio': {'array': List(Value('float32')), 'sampling_rate': Value('int64')}, 'label': Value('int64')}


In [16]:
from datasets import Value

result_ds[0] = result_ds[0].cast_column("label", Value("int64"))


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 643021/643021 [06:48<00:00, 1573.84 examples/s]


In [17]:
for ds in result_ds:
    print(ds.features)


{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}
{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}
{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': Value('int64')}
{'audio': {'array': List(Value('float32')), 'sampling_rate': Value('int64')}, 'label': Value('int64')}


In [ ]:
from datasets import Audio

In [19]:
tes_ds = result_ds[-1].cast_column("audio", Audio(decode=True))


In [22]:
parts = result_ds[:3] + [tes_ds]


In [23]:
ds_parts = [concatenate_datasets(parts[:2]), concatenate_datasets(parts[2:])]


In [24]:
print(len(ds_parts))


2


In [27]:
ds_parts[0].save_to_disk(str(local_tmp_dir) + "/part0", max_shard_size="200MB", num_proc=2)


Saving the dataset (2/228 shards):   1%|█▎                                                                                                                                                       | 5716/651589 [01:23<2:37:49, 68.21 examples/s]


FileNotFoundError: [Errno 2] No such file or directory

Process ForkPoolWorker-2:
Traceback (most recent call last):
  File "/home/nicolas/.local/lib/python3.13/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/nicolas/.local/lib/python3.13/site-packages/multiprocess/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nicolas/.local/lib/python3.13/site-packages/multiprocess/pool.py", line 114, in worker
    task = get()
  File "/home/nicolas/.local/lib/python3.13/site-packages/multiprocess/queues.py", line 370, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/home/nicolas/.local/lib/python3.13/site-packages/dill/_dill.py", line 311, in loads
    return load(file, ignore, **kwds)
  File "/home/nicolas/.local/lib/python3.13/site-packages/dill/_dill.py", line 297, in load
    return Unpickler(file, ignore=ignore, **kwds).load()
           ~~~~~~~~~~~~~~~~~~~~~~~

In [26]:
ds_parts[1].save_to_disk(str(local_tmp_dir) + "/part1")


Saving the dataset (104/104 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51042/51042 [20:30<00:00, 41.49 examples/s]


In [28]:
god_ds = concatenate_datasets(ds_parts)


In [29]:
num_shards=300


In [30]:
for i in range(num_shards):
    print(f"{i}/{num_shards}")
    shard = god_ds.shard(index=i, num_shards=num_shards, contiguous=True)
    shard.to_parquet(f"{LOCAL_DIR}/output/shard_{i:05d}.parquet")


0/300


Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  5.64ba/s]


1/300


Creating parquet from Arrow format: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.95ba/s]


2/300


...

Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.30ba/s]


298/300


Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.69ba/s]


299/300


Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.77ba/s]


In [31]:
testee = datasets.load_dataset("parquet", data_dir=f"{LOCAL_DIR}/output/")


Generating train split: 702631 examples [45:32, 257.18 examples/s] 


In [36]:
print(testee["train"][0]["audio"])


In [37]:
for i in range(len(testee["train"])):
    d = testee["train"][i]["audio"]


# Now we're ready for upload !

In [46]:
token = ""


In [47]:
api = HfApi(token=token)


In [50]:
from huggingface_hub import create_repo

def upload_directory(api, token, local_dir: str, repo_id: str, repo_type: str = "dataset"):
    """
    Uploads all files from `local_dir` to the Hugging Face Hub repository.
    Automatically skips already uploaded files (resumable).
    """

    create_repo(repo_id, repo_type=repo_type, token=token, exist_ok=True)

    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise ValueError(f"Local directory does not exist: {local_dir}")

    print(f"📂 Scanning directory: {local_dir}")
    files_to_upload = [p for p in local_dir.rglob("*") if p.is_file()]
    print(f"Found {len(files_to_upload)} files to check.")

    # 🧠 Get list of already uploaded files
    print(f"🔍 Fetching existing files in repo: {repo_id}")
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    print(f"Repo already has {len(existing_files)} files.")

    uploaded_count = 0
    skipped_count = 0

    for fpath in files_to_upload:
        # Normalize path in repo (relative to base directory)
        path_in_repo = str(fpath.relative_to(local_dir)).replace("\\", "/")

        if path_in_repo in existing_files:
            print(f"⏩ Skipping already uploaded: {path_in_repo}")
            skipped_count += 1
            continue

        try:
            print(f"⬆️ Uploading: {path_in_repo} ...")
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type=repo_type,
            )
            uploaded_count += 1
            print(f"✅ Uploaded: {path_in_repo}")
        except Exception as e:
            print(f"❌ Error uploading {fpath}: {e}")
            print("Stopping — rerun this script to resume.")
            break

    print("\n✅ Upload complete.")
    print(f"Uploaded: {uploaded_count}, Skipped: {skipped_count}")


In [ ]:
upload_directory(api, token, str(LOCAL_DIR) + "/output", REPO_ID, REPO_TYPE)


📂 Scanning directory: prepared_dataset/output
Found 300 files to check.
🔍 Fetching existing files in repo: Hibou-Foundation/datian
Repo already has 8 files.
⏩ Skipping already uploaded: shard_00253.parquet
⏩ Skipping already uploaded: shard_00093.parquet
⏩ Skipping already uploaded: shard_00039.parquet
⏩ Skipping already uploaded: shard_00012.parquet
⏩ Skipping already uploaded: shard_00098.parquet
⏩ Skipping already uploaded: shard_00202.parquet
⏩ Skipping already uploaded: shard_00211.parquet
⬆️ Uploading: shard_00275.parquet ...


Processing Files (0 / 0): |                                                                                                                                                                                        |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|                                                                                                                                                                                    |  550kB / 8.48GB,  393kB/s  
Processing Files (0 / 1):   0%|                                                                                                                                                                                    | 1.10MB / 8.48GB,  500kB/s  
Processing Files (0 / 1):   0%|                                                                                                                                                                                    | 1.62MB / 8.48GB,  677kB/s  
Processing Files (0 / 1):   0%|     

  2026-02-28T10:24:10.415079Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'zmq.error.ZMQError'>, value: ZMQError('Too many open files'), traceback: Some(<traceback object at 0x7f790f098ac0>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

